# Работа с данными бизнеса в ClickHouse

Проект стажировки с данными о чтении и прослушивании контента в сервисе Книги, где для поиска ответов на вопросы/предположениий продакт-менеджера рассчитываю метрики и изучаю паттерны использования сервиса. На основе расчетов - интерпретирую полученные результаты.

<b>Структура:</b>
</div>

- Задание 1 Активные города и регионы Топ-20

- Задание 2 Популярный контент Топ-5

- Задание 3 Топ-10 авторов книг

- Задание 4 Проверка гипотезы "iOS VS Android"

- Задание 5 Cвязь между форматом использования приложения и днём недели

- Задание 6 Проверка гипотезы "обновление iOS"

- Задание 7 Частота обновления приложения на каждой из платформ

- Задание 8 Подсчёт книг с тегом "Магия"

- Задание 9 Подсчёт книг со словом «магия» без тега "Магия"

- Задание 10 Проверка кол-ва категорий у каждой книги

- Задание 11 Поиск аномалий в длине пользовательской сессии

# Задание 1

Для начала продакт-менеджер хочет понять, где сервис пользуется наибольшей популярностью. Выведите топ-20 городов и регионов России по суммарному количеству прочитанных и прослушанных часов любого контента с мобильных устройств. Для каждой из платформ — iOS и Android — добавьте отдельный столбец с длительностью. Результат должен выглядеть так: город, общая длительность прочитанного и прослушанного контента, длительность на iOS, длительность на Android. Значения округлите до целых чисел для лучшей читаемости. Из выдачи также исключите федеральные округа — оставьте только города и области.

In [1]:
WITH russian_cities AS (
    SELECT
        usage_geo_id_name
    FROM source_db.audition
    WHERE usage_geo_id_name NOT LIKE '%округ%'
    GROUP BY usage_geo_id_name
)
SELECT
    a.usage_geo_id_name,
    -- Общая длительность по всем платформам
    round(sum(a.hours)) AS total_hours,
    -- Длительность по iOS
    round(sum(if(a.usage_platform_ru LIKE 'Букмейт iOS', a.hours, 0))) AS ios_hours,
    -- Длительность по Android
    round(sum(if(a.usage_platform_ru LIKE 'Букмейт Android', a.hours, 0))) AS android_hours
FROM source_db.audition a
JOIN russian_cities rc ON a.usage_geo_id_name = rc.usage_geo_id_name
WHERE usage_country_name = 'Россия' and usage_platform_ru IN ('Букмейт iOS', 'Букмейт Android')
GROUP BY a.usage_geo_id_name
ORDER BY total_hours DESC
LIMIT 20;

SyntaxError: invalid syntax (2565363845.py, line 1)

<b>Вывод по задаче 1:</b>
</div>
Топ-20:

| usage_geo_id_name                        | total_hours | ios_hours | android_hours |
|------------------------------------------|--------------|-----------|--------------|
| Москва                                   | 26629.0      | 8056.0    | 18573.0      |
| Санкт-Петербург                          | 12538.0      | 3794.0    | 8744.0       |
| Москва и Московская область              | 8248.0       | 2412.0    | 5837.0       |
| Екатеринбург                             | 4818.0       | 1493.0    | 3325.0       |
| Россия                                   | 4469.0       | 1320.0    | 3149.0       |
| Краснодар                                | 3998.0       | 1219.0    | 2779.0       |
| Новосибирск                              | 3824.0       | 1155.0    | 2669.0       |
| Ростов-на-Дону                           | 3115.0       | 964.0     | 2151.0       |
| Казань                                   | 3000.0       | 1022.0    | 1978.0       |
| Пермь                                    | 2914.0       | 821.0     | 2093.0       |
| Санкт-Петербург и Ленинградская область  | 2884.0       | 902.0     | 1982.0       |
| Уфа                                      | 2621.0       | 779.0     | 1842.0       |
| Нижний Новгород                          | 2602.0       | 702.0     | 1899.0       |
| Челябинск                                | 2586.0       | 735.0     | 1850.0       |
| Красноярск                               | 2126.0       | 685.0     | 1441.0       |
| Краснодарский край                       | 2084.0       | 675.0     | 1409.0       |
| Воронеж                                  | 1841.0       | 552.0     | 1290.0       |
| Тюмень                                   | 1827.0       | 581.0     | 1245.0       |
| Самара                                   | 1808.0       | 558.0     | 1250.0       |
| Нижегородская область                    | 1574.0       | 429.0     | 1145.0       |

Сервис пользуется наибольшей популярностью в Москве (26629 часов чтения и прослушиваний) и Санкт-Петербурге (12538 часов чтения и прослушиваний). Занятно, что во всех выделенных городах количество часов примерно в 2 раза Android выше, нежели на iOS. В конце топа с общим количеством часов по <2000 стоят Тюмень (1827 часов), Самара (1808 часов) и Нижегородская область (1574 часов)

# Задание 2


С активными регионами определились, а какой контент самый популярный? Получите топ-5 книг по суммарному количеству прочитанных и прослушанных часов на мобильных платформах. Также вычислите среднее время чтения и прослушивания в зависимости от типа книги: текст или аудио. Результат должен выглядеть так: название книги, её автор, суммарное время чтения и прослушивания, среднее время чтения текстовой книги, среднее время прослушивания аудиокниги.
В список включайте только те книги, которые используются в обоих форматах. Числовые значения округляйте до двух знаков после точки.

In [ ]:
WITH books_both_formats AS (
    SELECT
        main_content_id,
        main_content_name,
        main_author_name,
        main_content_type
    FROM source_db.content
    GROUP BY
        main_content_id,
        main_content_name,
        main_author_name,
        main_content_type
),
filtered_books AS (
    -- Отбираем только те книги, у которых есть оба формата
    SELECT
        main_content_name,
        main_author_name
    FROM books_both_formats
    GROUP BY
        main_content_name,
        main_author_name
    HAVING countDistinct(main_content_type) = 2
),
book_stats AS (
    SELECT
        c.main_content_name as main_content_name,
        c.main_author_name as main_author_name,
        round(sum(a.hours)) AS total_hours,
        round(avgIf(a.hours, c.main_content_type = 'Book'), 2) AS avg_read_time_text,
        round(avgIf(a.hours, c.main_content_type = 'Audiobook'), 2) AS avg_listen_time_audio
    FROM source_db.audition a
    JOIN books_both_formats c ON a.main_content_id = c.main_content_id
    JOIN filtered_books fb ON (c.main_content_name = fb.main_content_name)
        AND (c.main_author_name = fb.main_author_name)
    WHERE a.usage_platform_ru IN ('Букмейт iOS', 'Букмейт Android')
    GROUP BY c.main_content_name, c.main_author_name
)
SELECT
    main_content_name,
    main_author_name,
    total_hours,
    avg_read_time_text,
    avg_listen_time_audio
FROM book_stats
ORDER BY total_hours DESC
LIMIT 5;

<b>Вывод по задаче 2:</b>
</div>
Топ-5 книг по суммарному количеству прочитанных и прослушанных часов на мобильных платформах:

| main_content_name                                                     | total_hours | avg_read_time_text | avg_listen_time_audio |   |
|-----------------------------------------------------------------------|-------------|--------------------|-----------------------|---|
| Илон Маск                                                             | 1013.0      | 0.29               | 0.69                  |   |
| Железное пламя                                                        | 781.0       | 1.74               | 1.89                  |   |
| Убийства и кексики. Детективное агентство «Благотворительный магазин» | 542.0       | 0.68               | 1.63                  |   |
| Четвертое крыло                                                       | 502.0       | 1.58               | 1.34                  |   |
| Земля лишних. Трилогия                                                | 482.0       | 2.47               | 2.75               

Наибольшее среднее время чтения текстовой книги и аудиокниги у "Земля лишних. Трилогия" - 2.47 ч и 2.75, наименьшее 0.29 ч и 0.69 - у "Илон Маск".

# Задание 3


Составьте топ-10 авторов по суммарной длительности чтения их книг на всех платформах, включая веб. Для каждого автора добавьте количество уникальных текстовых книг (тип контента 'Book' ) и выведите среднюю длительность прослушивания их аудиокниг только на мобильных устройствах. Исключите авторов, у которых нет аудиокниг.

In [ ]:
WITH authors_books AS (
    SELECT
        c.main_author_name AS author,
        c.main_content_id AS book_id,
        c.main_content_type AS content_type
    FROM source_db.content c
),
author_read_time AS (
    SELECT
        ab.author as author,
        -- Общий время чтения книг
        sumIf(a.hours, c.main_content_type='Book') AS total_read_hours,
        -- Количество текстовых книг
        countDistinctIf(c.main_content_id, c.main_content_type='Book') AS text_books_count,
        -- Среднее время прослушивания аудиокниг на мобильных платформах
        round(avgIf(a.hours, c.main_content_type='Audiobook' AND a.usage_platform_ru IN ('Букмейт iOS', 'Букмейт Android')), 2) AS avg_audio_listen_time
    FROM source_db.audition a
    JOIN authors_books ab ON a.main_content_id = ab.book_id
    JOIN source_db.content c ON a.main_content_id = c.main_content_id
    WHERE a.usage_platform_ru IN ('Букмейт iOS', 'Букмейт Android', 'Букмейт Web')
    GROUP BY ab.author
    HAVING countIf(c.main_content_type='Audiobook') > 0
)
SELECT
    author,
    total_read_hours,
    text_books_count,
    avg_audio_listen_time
FROM author_read_time
ORDER BY total_read_hours DESC
LIMIT 10;

<b>Вывод по задаче 3:</b>
</div>
Топ-10 авторов по суммарной длительности чтения их книг на всех платформах, включая веб: 

|author|total_read_hours|text_books_count|avg_audio_listen_time|
|---|---|---|---|
| Александра Лисина | 1558.3513034871285 | 71 | 2.27 | 
| Дарья Донцова | 1498.9576724968501 | 163 | 1.91 | 
| Константин Муравьёв | 834.4004381813575 | 24 | 2.67 |
| Елена Звёздная | 814.9590558283962 | 43 | 1.69 | 
| Сергей Лукьяненко | 813.6173274006578 | 54 | 1.76 | 
| Робин Хобб | 687.2205103496672 | 18 | 1.29 | 
| Виктор Пелевин | 633.7019836705294 | 30 | 0.96 | 
| Ребекка Яррос | 565.9010513906542 | 2 | 1.67 |
| Татьяна Устинова | 545.4667478364427 | 63 | 1.53 | 
| Макс Фрай | 500.54441867151763 | 38 | 1.3 |

Наибольшее количество уникальных текстовых книг и общее время чтения книг у авторов -  Александра Лисина (71 книги, 1558.35 ч) и Дарья Донцова (163 книги, 1498.96 ч), где средняя длительность прослушивания аудиокниг только на мобильных устройствах - 2.27 ч и 1.91 ч. Наименьшее количество уникальных текстовых книг и общее время чтения книг у авторов - Татьяна Устинова (63 книги, 545.47 ч) и Макс Фрай (38 книг, 500.54 ч), где средняя длительность прослушивания аудиокниг только на мобильных устройствах - 1.53 ч и 1,3 ч.

# Задание 4



У продакт-менеджера есть предположение, что среди Android-пользователей аудиокниги почти так же популярны, как тексты. А среди iOS-пользователей читателей книг вдвое больше, чем слушателей, если считать по суммарной длительности сессии.
Проверьте предположение менеджера. Для начала выделите три сегмента пользователей:
«Слушатель» — тот, кто преимущественно пользуется аудиокнигами. Прослушивание книг составляет 70% и выше от суммарной длительности сессий.
«Читатель» — преимущественно пользуется текстовыми книгами. Чтение книг — от 70%.
«Оба» — остальные пользователи сервиса.
Исключите пользователей, у которых нет сессий ни с книгами, ни с аудиокнигами, и посчитайте количество пользователей в каждом из сегментов.
На основе полученных данных проверьте предположение менеджера о том, что среди пользователей Android примерно одинаковое количество читателей и слушателей, а на устройствах iOS читателей книг вдвое больше, чем слушателей. Чтобы определить основную платформу пользователя, учитывайте время её использования. Например, если пользователь посещал сервис с двух устройств: два часа на iOS и пять часов на Android, то основной платформой такого пользователя будет Android.

In [ ]:
WITH user_platform_time AS (
    SELECT
        a.puid,
        sumIf(a.hours, a.usage_platform_ru LIKE '%iOS%') AS ios_time,
        sumIf(a.hours, a.usage_platform_ru LIKE '%Android%') AS android_time
    FROM source_db.audition a
    GROUP BY a.puid
),
user_main_platform AS (
    SELECT
        puid,
        multiIf(
            ios_time > android_time, 'iOS',
            android_time > ios_time, 'Android',
            'Other'
        ) AS main_platform
    FROM user_platform_time
),
user_content_time AS (
    SELECT
        a.puid,
        sumIf(a.hours, c.main_content_type='Book') AS book_time,
        sumIf(a.hours, c.main_content_type='Audiobook') AS audiobook_time
    FROM source_db.audition a
    JOIN source_db.content c ON a.main_content_id = c.main_content_id
    GROUP BY a.puid
),
user_segments AS (
    SELECT
        uct.puid,
        ump.main_platform,
        (uct.book_time + uct.audiobook_time) AS total_time,
        if((uct.book_time + uct.audiobook_time) = 0, NULL, uct.book_time / (uct.book_time + uct.audiobook_time)) AS reading_ratio,
        if((uct.book_time + uct.audiobook_time) = 0, NULL, uct.audiobook_time / (uct.book_time + uct.audiobook_time)) AS audio_ratio
    FROM user_content_time uct
    JOIN user_main_platform ump ON uct.puid = ump.puid
)
SELECT
    main_platform,
    countIf(reading_ratio >= 0.7) AS readers,
    countIf(audio_ratio >= 0.7) AS listeners,
    -- "Оба" — это те, у кого оба показателя < 0.7
    countIf(reading_ratio < 0.7 AND audio_ratio < 0.7 AND total_time > 0) AS both,
    count(*) AS total_users
FROM user_segments
WHERE total_time > 0
GROUP BY main_platform;

<b>Вывод по задаче 4:</b>

|main_platform|readers|listeners|both|total_users|
|-------------|-------|---------|----|-----------|
|iOS          |1770   |5683     |356 |7809       |
|Other        |459    |13428    |9   |13896      |
|Android      |2138   |6599     |474 |9211       |

Предположение, что среди Android-пользователей аудиокниги почти так же популярны, как тексты - подтвердилось. А среди iOS-пользователей читателей книг вдвое больше, чем слушателей, если считать по суммарной длительности сессии - не подтвердилось.

# Задание 5



Изучите, существует ли связь между форматом использования приложения (прослушивание или чтение) и днём недели. Падает ли использование аудиокниг в выходные на всех платформах, включая веб? Чтобы это узнать, для каждого типа контента посчитайте среднее время его использования в рабочие и выходные дни и округлите до целого числа. Используя оконные функции, вы также можете пользоваться и комбинаторами.

In [ ]:
WITH user_content_time AS (
    SELECT
        a.audition_id,
        c.main_content_type,
        a.hours,
        a.msk_business_dt_str,
        a.usage_platform_ru
    FROM source_db.audition a
    JOIN source_db.content c ON a.main_content_id = c.main_content_id
    WHERE
        c.main_content_type IN ('Book', 'Audiobook') -- фильтр по типам контента
        AND a.usage_platform_ru IN ('Букмейт iOS', 'Букмейт Android', 'Букмейт Web') -- фильтр по платформам
),
content_day_week AS (
    SELECT
        main_content_type,
        -- Получение дня недели: 0=понедельник, 6=воскресенье
        toDayOfWeek(msk_business_dt_str) - 1 AS day_of_week,
        hours
    FROM user_content_time
),
day_category AS (
    SELECT
        main_content_type,
        -- Категория дня: будни (0-4), выходные (5-6)
        CASE WHEN day_of_week IN (5,6) THEN 'выходные' ELSE 'будни' END AS day_type,
        hours
    FROM content_day_week
),
avg_hours_by_type AS (
    SELECT
        main_content_type,
        day_type,
        ROUND(AVG(hours)) AS avg_hours -- округление до целого
    FROM day_category
    GROUP BY main_content_type, day_type
)
SELECT
    main_content_type,
    day_type,
    avg_hours
FROM avg_hours_by_type
ORDER BY main_content_type, day_type;

<b>Вывод по задаче 5:</b>

|main_content_type|day_type  |avg_hours|
|-----------------|----------|---------|
| Audiobook       | будни    | 1.0     |
| Audiobook       | выходные | 1.0     |
| Book            | будни    | 1.0     |
| Book            | выходные | 1.0     | 

Исходя из полученных данных - не существует связь между форматом использования приложения (прослушивание или чтение) и днём недели. В среднем использование аудиокниг в выходные сумарно на всех платформах, включая веб, - равно 1. Меня смутил данный результа, а потому заменила основную часть кода - убрала округление до целого и посчитала строки.

In [ ]:
SELECT
    main_content_type,
    day_type,
    AVG(hours) AS avg_hours,
    COUNT(*) AS count_rows
FROM day_category
GROUP BY main_content_type, day_type;

<b>Дополнение по задаче 5:</b>

|main_content_type|day_type  |avg_hours|count_rows|
|---|---|---|---|
| Audiobook | выходные | 1.3450710854778118 | 25220 |  
| Book | выходные | 0.8555424724056732 | 25336 |  
| Audiobook | будни | 1.2868333371109502 | 79883 |  
| Book | будни | 0.789620708691695 | 72468 |  

Похоже действительно средние значения близки. В выходные среднее время прослушиваний и чтений немного больше, но в данном задании получается, что это не настолько значимо.

# Задание 6


Продакт-менеджер хочет отслеживать обновления приложений на Android и iOS. У него есть предположение, что больший процент пользователей iOS используют последнюю версию приложения и в целом чаще его обновляют.
Для начала изучите, у какой части пользователей на текущий момент стоят последние версии приложения на каждой из платформ. Для этого посчитайте последнюю активную версию каждого пользователя и сравните её с последней версией у каждой платформы. Для каждой платформы выведите процент пользователей с последней версией приложения и округлите его до двух знаков после точки.

In [ ]:
WITH latest_user_versions AS (
    SELECT
        puid,
        usage_platform_ru,
        -- Находим последнюю активность по дате
        argMax(app_version, msk_business_dt_str) AS last_version
    FROM source_db.audition
    WHERE usage_platform_ru IN ('Букмейт iOS', 'Букмейт Android') -- фильтр по платформам
    GROUP BY puid, usage_platform_ru
),
user_platform_count AS (
    SELECT
        puid,
        -- Количество уникальных платформ, которые использует пользователь
        countDistinct(usage_platform_ru) AS platform_count
    FROM source_db.audition
    WHERE usage_platform_ru IN ('Букмейт iOS', 'Букмейт Android') -- фильтр по платформам
    GROUP BY puid
),
latest_platform_versions AS (
    SELECT
        usage_platform_ru,
        -- Максимальная версия для платформы
        max(app_version) AS max_version
    FROM source_db.audition
    WHERE usage_platform_ru IN ('Букмейт iOS', 'Букмейт Android') -- фильтр по платформам
    GROUP BY usage_platform_ru
),
user_with_platform_flag AS (
    SELECT
        luv.puid,
        luv.usage_platform_ru,
        luv.last_version,
        lpv.max_version,
        -- Проверка, использует ли пользователь последнюю версию
        (luv.last_version = lpv.max_version) AS is_latest
    FROM latest_user_versions AS luv
    JOIN latest_platform_versions AS lpv
        ON luv.usage_platform_ru = lpv.usage_platform_ru
),
users_single_platform AS (
    SELECT
        uwp.*
    FROM user_with_platform_flag uwp
    JOIN user_platform_count upc
        ON uwp.puid = upc.puid
    WHERE upc.platform_count = 1 -- оставляем только пользователей, использующих одну платформу
),
platform_stats AS (
    SELECT
        usage_platform_ru,
        countIf(is_latest) AS users_with_latest_version,
        count() AS total_users
    FROM users_single_platform
    GROUP BY usage_platform_ru
)
SELECT
    usage_platform_ru,
    -- Процент пользователей с последней версией, округленный до 2 знаков
    round((users_with_latest_version * 100.0) / total_users, 2) AS percentage_with_latest
FROM platform_stats;

<b>Вывод по задаче 6:</b>

|usage_platform_ru|percentage_with_latest|
|-----------------|----------------------|
| Букмейт iOS     |                  1.9 |
| Букмейт Android |                29.35 | 

Предположение, что больший процент пользователей iOS используют последнюю версию приложения и в целом чаще его обновляют - не подтвердилось, так как по результату вычислений всё наоборот - Android-пользователи чаще используют последнюю версию.


# Задание 7


Теперь продакт-менеджер хочет понять, как часто пользователи обновляют приложение на каждой из платформ. Фактом обновления считайте изменение версии у каждого пользователя. Представьте, что любое изменение возможно только в сторону более новой версии.
Проверьте предположение о том, что пользователи iOS чаще обновляют приложение. Посчитайте метрику update_rate, которая покажет среднюю частоту обновлений на пользователя. Округлите её до двух знаков после точки.

In [3]:
WITH source AS (
    SELECT
        puid,
        usage_platform_ru,
        msk_business_dt_str AS event_time, -- если это уже Date
        app_version
    FROM source_db.audition
),
user_version_changes AS (
    SELECT
        puid,
        usage_platform_ru,
        event_time,
        app_version,
        lagInFrame(app_version, 1) OVER (PARTITION BY puid, usage_platform_ru ORDER BY event_time) AS prev_version
    FROM source
),
user_update_counts AS (
    SELECT
        puid,
        usage_platform_ru,
        countIf(app_version != prev_version AND prev_version IS NOT NULL) AS total_version_changes
    FROM user_version_changes
    GROUP BY puid, usage_platform_ru
),
platform_stats AS (
    SELECT
        usage_platform_ru,
        sum(total_version_changes) AS total_changes,
        count(DISTINCT puid) AS user_count
    FROM user_update_counts
    WHERE usage_platform_ru IN ('Букмейт iOS','Букмейт Android') -- условие фильтрации
    GROUP BY usage_platform_ru
),
update_rate AS (
    SELECT
        usage_platform_ru,
        round(total_changes * 1.0 / user_count, 2) AS update_rate
    FROM platform_stats
)
SELECT
    usage_platform_ru,
    update_rate
FROM update_rate
ORDER BY usage_platform_ru;

SyntaxError: invalid syntax (1883067610.py, line 1)

<b>Вывод по задаче 7:</b>

|usage_platform_ru|update_rate|
|---|---|
| Букмейт Android|3.18|
|Букмейт iOS|2.43|

Предположение о том, что пользователи iOS чаще обновляют приложения не оправдалось. Средняя частота обновлений на пользователя в Букмейт iOS - 2.43, в противовес 3.18 у Букмейт Android.


# Задание 8


Новая задача — у коллег есть опасения, что не все книги на тему магии верно размечены с точки зрения категорий.  Считается, что у книги должно быть не больше 3–4 категорий с темами. Необходимо найти все книги на магическую тему, которые при этом не входят в художественную литературу, и проверить, правильно ли они размечены.
Начните с подсчёта книг с тегом «Магия». Выведите количество таких книг в каталоге.

In [ ]:
SELECT
    count() AS magic_non_fiction_content_count
FROM
    source_db.content
WHERE
    has(published_topic_title_list, 'Магия');

<b>Вывод по задаче 8:</b>
Количество книг с тегом «Магия» - 46 шт

# Задание 9


Найдите все книги со словом «магия» в названии, для которых не проставлен тег «Магия». При этом не учитывайте книги с тегом «Художественная литература». Выведите количество таких книг в каталоге.

In [ ]:
SELECT
    count() AS books_with_magica_in_title_without_magic_tag
FROM
    source_db.content
WHERE
    main_content_name ILIKE '%магия%'
    AND NOT has(published_topic_title_list, 'Магия')
    AND NOT has(published_topic_title_list, 'Художественная литература');

<b>Вывод по задаче 8:</b>
Количество книг - 49 шт

# Задание 10


Посчитайте среднее количество категорий у книг с тегом «Магия» и среднее количество категорий у книг в каталоге в целом. Округлите значения до двух знаков после точки. Напомним, что коллегам важно, чтобы у каждой книги было не больше 3–4 категорий. Получится ли не превысить рекомендованного количества?

In [ ]:
WITH
    -- Посчитать количество категорий у каждой книги, сохраняя список тегов
    content_categories AS (
        SELECT
            main_content_id,
            length(published_topic_title_list) AS category_count,
            published_topic_title_list  -- добавляем для условия
        FROM
            source_db.content
    ),
    -- Среднее количество категорий у книг с тегом «Магия»
    avg_categories_magic AS (
        SELECT
            round(avgIf(category_count, has(published_topic_title_list, 'Магия')), 2) AS avg_categories_with_magic
        FROM
            content_categories
    ),
    -- Среднее количество категорий у всех книг
    avg_categories_all AS (
        SELECT
            round(avg(category_count), 2) AS avg_categories_in_catalog
        FROM
            content_categories
    )
SELECT
    (SELECT avg_categories_with_magic FROM avg_categories_magic) AS avg_categories_with_magic,
    (SELECT avg_categories_in_catalog FROM avg_categories_all) AS avg_categories_in_catalog;

<b>Вывод по задаче 10:</b>
Среднее количество категорий у книг с тегом «Магия» - 3.22,	а среднее количество категорий у книг в каталоге в целом - 3.77. По проекту желательно, чтобы у каждой книги было не больше 3–4 категорий, значит в целом не превышено рекомендованное количество.

# Задание 11



Продакт-менеджер выяснил, что в приложении одной из мобильных платформ могла возникнуть проблема — длина пользовательской сессии (поле hours_sessions_long ) записывается некорректно, и это происходит как минимум в одной из стран. Чтобы найти аномалию в данных, используйте такую меру дисперсии как коэффициент вариации. Напомним его формулу: коэффициент определяется как отношение стандартного отклонения к среднему. Чем выше этот показатель, тем более подозрительно с точки зрения анализа распределены данные.
Исследуйте коэффициент по странам и мобильным платформам. В какой стране и на какой платформе видна аномалия в данных? Ограничьте выборку одной страной, в которой коэффициент вариации для одной из платформ будет наибольшим.

In [ ]:
WITH
    -- Вычисляем показатели для каждой страны и платформы
    stats AS (
        SELECT
            usage_country_name,
            usage_platform_ru,
            avg(hours_sessions_long) AS mean_hours,
            stddevSamp(hours_sessions_long) AS stddev_hours,
            -- Вычисляем коэффициент вариации
            if(avg(hours_sessions_long) = 0, 0, stddevSamp(hours_sessions_long) / avg(hours_sessions_long)) AS coef_variation
        FROM
            source_db.audition
        WHERE
            usage_platform_ru IN ('Букмейт iOS', 'Букмейт Android')
        GROUP BY
            usage_country_name,
            usage_platform_ru
    ),
    -- Находим максимальный коэффициент вариации среди всех пар страна-платформа
    max_coef AS (
        SELECT
            max(coef_variation) AS max_cv
        FROM
            stats
    ),
    -- Выбираем страну и платформу с максимальным коэффициентом вариации
    anomaly AS (
        SELECT
            usage_country_name,
            usage_platform_ru,
            coef_variation
        FROM
            stats
        WHERE
            coef_variation = (SELECT max_cv FROM max_coef)
        LIMIT 1
    )
SELECT
    anomaly.usage_country_name AS country,
    anomaly.usage_platform_ru AS platform,
    anomaly.coef_variation AS coefficient_of_variation
FROM
    anomaly;

<b>Вывод по задаче 11:</b>
На Android платформе видна аномалия в данных в Латвии, где наибольший коэффициент вариации - 7.8.